In [1]:
source(here::here('MG3.functions1.R'))

# Check the HMM hits across the genomes

## Generating a swarm file with line for each genome accession

In [2]:
# Get list of the genomes
assembyfilename=paste0(StaticInpDir,"assemblies.json")
assembltext=readLines(assembyfilename)
assembliesJSON <- fromJSON(assembltext)

GenomesDF=NULL
for (assembly in assembliesJSON$reports ){
    row=list(accession=assembly$accession, sample= assembly$assembly_info$biosample$accession,
            taxon= assembly$assembly_info$biosample$description$organism$tax_id)
    if (is.null(row$taxon)){
        row$taxon=assembly$organism$tax_id
    }
    if (is.null(row$sample)){
        row$sample=NA
    }
    if (is.null(GenomesDF)){
        GenomesDF=as.data.frame(row)
    }else{
        GenomesDF=rbind(GenomesDF, as.data.frame(row))
    }
}

In [3]:
# create dir
outputDirTMP=paste0(OutputDir,'HMMsAllSpeciesResults/')
comToBash=paste0("mkdir -p ",OutputDir,"HMMsAllSpeciesResults/slurm_logs")
system(comToBash, intern = FALSE, wait = TRUE)


In [4]:
# raw
dbGetQuery(mydbRO, "SELECT COUNT(DISTINCT ensembl_gene_id) FROM BijectiveRelevantSpeciesView LIMIT 5;")

COUNT(DISTINCT ensembl_gene_id)
<int>
9847


In [5]:
# context checked
dbGetQuery(mydbRO, "SELECT COUNT(DISTINCT ensembl_gene_id) FROM positively_ided_using_context LIMIT 5;")

COUNT(DISTINCT ensembl_gene_id)
<int>
9608


In [6]:
sbatchscript="~/data/lambda/notebooks/runHMMsearchSpeciev2.sh"

max_chunk=2500
fs=list.files(paste0(OutputDir,"HMMs/"), pattern = "ENSG.*MSA.faa.hmm$", full.names = TRUE)
# filtering the HMM ids to HomologyBijectiveWellDefined
HMMsToTest=dbGetQuery(mydbRO, "SELECT DISTINCT ensembl_gene_id FROM BijectiveRelevantSpeciesView")[[1]] 
                            # we are using the bijective view since not positively IDed HMMs can still be good neighbors
fs=fs[ gsub(pattern = ".faa.MSA.faa.hmm", replacement = "", x = basename(fs), fixed = TRUE) %in% HMMsToTest ]

startpos=1
i=0
swarmlines=NULL

while (startpos<length(fs)){
    endpos=min(startpos+max_chunk,length(fs))
    fname=paste0(OutputDir,"/HMMsAllSpeciesResults/filelist-",i,".txt")
    writeLines(text = fs[startpos:endpos], 
               con=fname)

    for (accession in GenomesDF$accession){
        swarmline=paste0(sbatchscript," ", accession, " ", StaticInpDir," ", 
                         fname," ", maxPvalue, " ", OutputDir,"/HMMsAllSpeciesResults/",accession,"/")
        swarmlines=addToVec(swarmlines, swarmline)
    }
    
    startpos=endpos+1
    i=i+1
}

In [7]:
swarmparams=paste0("#SWARM -t 1 -g 5 --time=24:00:00 --gres=lscratch:10 --logdir ", path.expand(OutputDir),"HMMsAllSpeciesResults/slurm_logs/")
maxchunk=1000
timestamp=getTimeStamp()
processed=0
i=0
swarmFiles=NULL
while (processed<length(swarmlines)){
    upperlimit=processed+min(maxchunk, length(swarmlines)-processed)
    message(paste(processed+1, upperlimit))
    fname=paste0(OutputDir,"HMMsAllSpeciesResults/runningHMMsearch.",i,"-", timestamp,".swarm")
    writeLines(text = c(swarmparams, swarmlines[(processed+1):upperlimit]), 
               con=fname)
    message("Run from bash: ", fname)
    swarmFiles=addToVec(swarmFiles, fname)
    processed=upperlimit
    i=i+1
}

1 892

Run from bash: ~/data/lambda/output/HMMsAllSpeciesResults/runningHMMsearch.0-2026-01-28_17.31.28.41278.swarm



## Veryfying if the work is done

In [3]:
HMMsToTest=dbGetQuery(mydbRO, "SELECT DISTINCT ensembl_gene_id FROM BijectiveRelevantSpeciesView")[[1]]


In [4]:
notFoundFilesDF=NULL
for (accession in GenomesDF$accession){
    tmpdir=paste0(OutputDir,'HMMsAllSpeciesResults/', accession)
    fs=list.files(tmpdir, pattern = "ENSG.*.faa.MSA.faa.hmm.out$", full.names = TRUE)
    foundHMMs=gsub(pattern = ".faa.MSA.faa.hmm.out", replacement = "", x = basename(fs), fixed = TRUE) 
    problemHMMs=setdiff(HMMsToTest, foundHMMs)
    if (length(problemHMMs)>0){
        tmpDF=data.frame(accession=accession, HMM=problemHMMs)
        notFoundFilesDF=addToDF(notFoundFilesDF, tmpDF)
    }
}

In [5]:
notFoundFilesDF

NULL

In [7]:
length(HMMsToTest)

[1] 9847

In [8]:
length(foundHMMs)

[1] 9847